In [1]:
import sqlite3
import pandas as pd
import plotly.express as px

conn = sqlite3.connect("voos.db")

In [4]:
def query(arquivo_sql):
    """Lê um arquivo .sql e executa no banco."""
    sql = open(arquivo_sql, encoding="utf-8").read()
    return pd.read_sql(sql, conn)

In [5]:
# --- Análise 1: atraso por companhia ---
df_cia = query("queries/02_atraso_por_companhia.sql")

In [7]:
fig = px.bar(
    df_cia.head(15),
    x="companhia", y="atraso_medio_min",
    color="voos_criticos",
    title="Atraso médio por companhia aérea - 2024",
    labels={"atraso_medio_min": "Atraso médio (min)", "companhia": "Companhia"}
)
fig.show()

In [8]:
# --- Análise 2: evolução mensal ---
df_mes = query("queries/04_atraso_por_mes.sql")

In [9]:
fig2 = px.line(
    df_mes, x="mes", y="atraso_medio_min",
    title="Evolução do atraso médio ao longo de 2024",
    markers=True
)
fig2.show()

In [10]:
# --- Análise 3: distribuição de atrasos (log para reduzir skew) ---
import numpy as np

In [11]:
df_voos = pd.read_sql("""
    SELECT atraso_partida_min FROM voos
    WHERE situação_voo = 'REALIZADO'
    AND atraso_partida_min BETWEEN 1 AND 600
""", conn)

In [12]:
df_voos["log_atraso"] = np.log1p(df_voos["atraso_partida_min"])

In [14]:
fig3 = px.histogram(
    df_voos, x="log_atraso", nbins=80,
    title="Distribuição de atrasos (escala logarítmica)"
)
fig3.show()